---
#LangChain을 활용한 AI 자동화


---

# 환경 구축


- 구글 드라이브 연동

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/Colab Notebooks/LangChain

/content/drive/MyDrive/Colab Notebooks/LangChain


## 필수 라이브러리 설치

In [3]:
!pip install -qU langchain>=1.0.0 langchain_community

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [4]:
!pip install -qU langchain-openai langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.1 MB/s eta 0:00:00


In [5]:
# langchain-chroma, faiss-cpu : 벡터 DB를 오픈소스로 제공
!pip install -qU langchain-chroma faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

In [6]:
import os

from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
# 다양한 프로바이더 모델들을 초기화해주는 기능
from langchain.chat_models import init_chat_model

from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import FewShotPromptTemplate
from langchain_core.prompts import FewShotChatMessagePromptTemplate
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate

# Runnable 인스턴스 생성 (체인 연결이 가능한 형태로 변환)
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables import RunnableParallel
from langchain_core.runnables import RunnableLambda

from langchain_core.output_parsers import StrOutputParser

# stream 기능
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
# 유사도가 높은 예시를 선택
from langchain_core.example_selectors import SemanticSimilarityExampleSelector

# 벡터 DB
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores import Chroma

# 사용한 토큰량, 요금을 반환하는 함수
from langchain_community.callbacks import get_openai_callback

# 마크다운 형태로 출력
from IPython.display import display, Markdown

# 구조화된 출력을 위한 기능
from pydantic import BaseModel, Field

/tmp/ipykernel_7449/2029655133.py:27: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [7]:
# 파일에서 API 키 읽기
with open("./key/openai_key.txt", "r") as f:
    api_key = f.read().strip()

os.environ['OPENAI_API_KEY'] = api_key

In [8]:
# 파일에서 API 키 읽기
with open("./key/google_key.txt", "r") as f:
    api_key = f.read().strip()

os.environ['GOOGLE_API_KEY'] = api_key

# FewShotPromptTemplate

In [9]:
# (1) 몇 개의 예시를 작성 (Input의 반대말을 Output으로 출력하는 7개 예시)
examples1 = [
    {"input": "happy", "output": "sad"},
    {"input": "tall", "output": "short"},
    {"input": "sunny", "output": "rainy"},
    {"input": "surprised", "output": "calm"},
    {"input": "dry", "output": "humid"},
    {"input": "hot", "output": "cold"},
    {"input": "satisfied", "output": "dissatisfied"}
]

In [10]:
llm = init_chat_model("gpt-4o-mini")

# 예시들을 등록하는 프롬프트
prompt = PromptTemplate.from_template("Input: {input}\nOutput: {output}")

# 퓨삿 프롬프트 설정
fewshot_prompt = FewShotPromptTemplate(
    # 사용할 예시
    examples=examples1,
    # 프롬프트
    example_prompt=prompt,
    # 프롬프트의 마지막에 포함되는 내용
    suffix="Input: {input}\nOutput:",
    # 사용자에게 입력받을 변수
    input_variables=["input"]
)

print(fewshot_prompt.format(input="여름"))

Input: happy
Output: sad

Input: tall
Output: short

Input: sunny
Output: rainy

Input: surprised
Output: calm

Input: dry
Output: humid

Input: hot
Output: cold

Input: satisfied
Output: dissatisfied

Input: 여름
Output:


In [11]:
chain = fewshot_prompt | llm | StrOutputParser()
print(chain.invoke({"input": "여름"}))
print(chain.invoke({"input": "夜"}))

겨울
日


In [12]:
examples2 = [
    {
        "question":"해도는 빨강색을 좋아하고 병관은 파란색을 좋아한다.",
        "answer":'"해도":"빨강색", "병관":"파란색"'
    },
    {
        "question":"길동은 짜장을 좋아하고, 심청은 짬뽕을 좋아한다.",
        "answer":'"길동":"짜장","심청":"짬뽕"'
    },
    {
        "question":"영표는 국어를 좋아하고 세연은 수학을 좋아한다.",
        "answer":'"영표":"국어", "세연":"수학"'
    },
    {
        "question":"명진은 웹을 강의하고 보미는 사물인터넷을 강의한다.",
        "answer":'"명진":"웹","보미":"사물인터넷"'
    },
    {
        "question":"병관는 서울에 살고, 해도는 광주에 산다.",
        "answer":'"병관":"서울", "해도":"광주"'
    }
]

In [13]:
llm = init_chat_model("gpt-4o-mini")

# 예시들을 등록하는 프롬프트
prompt = PromptTemplate.from_template("question: {question}\nanswer: {answer}")

# 퓨삿 프롬프트 설정
fewshot_prompt = FewShotPromptTemplate(
    # 사용할 예시
    examples=examples2,
    # 프롬프트
    example_prompt=prompt,
    # 프롬프트의 마지막에 포함되는 내용
    suffix="question: {question}\nanswer:",
    # 사용자에게 입력받을 변수
    input_variables=["question"]
)

print(fewshot_prompt.format(question="사과는 한박스에 3만원이고 수박은 하나에 3만원이다"))

question: 해도는 빨강색을 좋아하고 병관은 파란색을 좋아한다.
answer: "해도":"빨강색", "병관":"파란색"

question: 길동은 짜장을 좋아하고, 심청은 짬뽕을 좋아한다.
answer: "길동":"짜장","심청":"짬뽕"

question: 영표는 국어를 좋아하고 세연은 수학을 좋아한다.
answer: "영표":"국어", "세연":"수학"

question: 명진은 웹을 강의하고 보미는 사물인터넷을 강의한다.
answer: "명진":"웹","보미":"사물인터넷"

question: 병관는 서울에 살고, 해도는 광주에 산다.
answer: "병관":"서울", "해도":"광주"

question: 사과는 한박스에 3만원이고 수박은 하나에 3만원이다
answer:


In [ ]:
chain = fewshot_prompt | llm | StrOutputParser()
print(chain.invoke({"question": "두부 한모는 한개이고 고등어 한손은 두마리다"}))

"두부":"1개", "고등어":"2마리"


# Callbacks - 스트리밍(streaming)

In [14]:
llm = init_chat_model("gpt-4o-mini",
                      streaming=True,
                      callbacks=[StreamingStdOutCallbackHandler()]
                      )
chain = llm | StrOutputParser()

answer = chain.invoke("RAG에 대해 120자마다 줄을 바꿔서 500자 내외로 설명해줘")


RAG는 "Retrieval-Augmented Generation"의 약자로,  

정보 검색과 생성 모델을 결합한 방법론입니다.  

주로 대화형 AI나 질문 응답 시스템에서 활용되며,  

기존의 언어 모델이 가지는 한계를 보완하는 데 도움을 줍니다.  

RAG는 두 가지 주요 구성 요소로 이루어져 있습니다.  

첫째, 정보 검색기(retriever)는 관련 문서를 검색하고,  

둘째, 생성기(generator)는 이 정보를 기반으로  

맥락에 맞는 응답을 생성합니다.  

이 방식은 모델이 사전에 학습한 데이터 외에도  

실시간으로 필요한 정보를 활용할 수 있도록 하여  

정확성과 관련성을 높이는 데 기여합니다.  

RAG는 특히 특정 도메인에 대한 지식을 요구하는  

질문에 대해 유용하며, 다양한 산업에서사용되고 있습니다.  

예를 들어, 고객 지원 또는 의료 분야에서   

정확한 정보를 제공하는 데 많이 적용됩니다.  

이러한 통합 접근 방식으로 인해 RAG는  

모델의 신뢰성을 심화시키고, 사용자 경험을 개선하며,  

더 나아가 복잡한 질문에도 효과적으로 대응할 수 있습니다.  

In [15]:
examples3 = [
    {
        "question": "스티브 잡스와 아인슈타인 중 누가 더 오래 살았나요?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 스티브 잡스는 몇 살에 사망했나요?
중간 답변: 스티브 잡스는 56세에 사망했습니다.
추가 질문: 아인슈타인은 몇 살에 사망했나요?
중간 답변: 아인슈타인은 76세에 사망했습니다.
최종 답변은: 아인슈타인
"""},
    {
        "question": "네이버의 창립자는 언제 태어났나요?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 네이버의 창립자는 누구인가요?
중간 답변: 네이버는 이해진에 의해 창립되었습니다.
추가 질문: 이해진은 언제 태어났나요?
중간 답변: 이해진은 1967년 6월 22일에 태어났습니다.
최종 답변은: 1967년 6월 22일
"""},
    {
        "question": "율곡 이이의 어머니가 태어난 해의 통치하던 왕은 누구인가요?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 율곡 이이의 어머니는 누구인가요?
중간 답변: 율곡 이이의 어머니는 신사임당입니다.
추가 질문: 신사임당은 언제 태어났나요?
중간 답변: 신사임당은 1504년에 태어났습니다.
추가 질문: 1504년에 조선을 통치한 왕은 누구인가요?
중간 답변: 1504년에 조선을 통치한 왕은 연산군입니다.
최종 답변은: 연산군
"""},
    {
        "question": "올드보이와 기생충의 감독이 같은 나라 출신인가요?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 올드보이의 감독은 누구인가요?
중간 답변: 올드보이의 감독은 박찬욱입니다.
추가 질문: 박찬욱은 어느 나라 출신인가요?
중간 답변: 박찬욱은 대한민국 출신입니다.
추가 질문: 기생충의 감독은 누구인가요?
중간 답변: 기생충의 감독은 봉준호입니다.
추가 질문: 봉준호는 어느 나라 출신인가요?
중간 답변: 봉준호는 대한민국 출신입니다.
최종 답변은: 예
"""}
]

# ExampleSelector

In [16]:
llm = init_chat_model(model = "gpt-4o-mini")

# OpenAIEmbeddings() : 사용자 질문과 예시를 임베딩할 모델
# FAISS : 사용할 벡터 DB
# k=2 : 사용할 유사 예시의 갯수
example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples1, OpenAIEmbeddings(), FAISS, k=2
)

prompt = PromptTemplate.from_template("Input: {input}\nOutput: {output}")

fewshot_prompt = FewShotPromptTemplate(
    # 전체 예시 대신에 선택한 예시만 등록
    example_selector=example_selector,
    example_prompt=prompt,
    suffix="Input: {input}\nOutput:",
    input_variables=["input"]
)

print(fewshot_prompt.format(input="여름"))

Input: hot
Output: cold

Input: sunny
Output: rainy

Input: 여름
Output:


In [ ]:
chain = fewshot_prompt | llm | StrOutputParser()
print(chain.invoke({"input": "여름"}))
print(chain.invoke({"input": "눈"}))

겨울
비


- 질문 : "Google이 창립된 연도에 Bill Gates의 나이는 몇살인가요?"
- 사용할 예시 : examples3
- 선택할 예시 갯수 : 1개

In [17]:
llm = init_chat_model(model = "gpt-5.4")

example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples3, OpenAIEmbeddings(), FAISS, k=1
)

prompt = PromptTemplate.from_template("input: {question}\noutput: {answer}")

fewshot_pormpt = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=prompt,
    suffix = "input : {question}\noutput:",
    input_variables=["question"]
)

chain = fewshot_prompt | llm | StrOutputParser()

print(chain.invoke({"input": "Google이 창립된 연도에 Bill Gates의 나이는 몇살인가요?"}))


43살


# MessagePromptTemplate 활용

In [23]:
llm = init_chat_model(model="gpt-5.4", temperature=0, max_tokens=500)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 천문학과 관련된 지식을 답변하는 전문 에이전트이다."),
        ("human", "{user_input}")
    ]
)

messages = prompt.format_messages(user_input = "태양계에서 가장 큰 행성은 무엇인가요?")

messages

[SystemMessage(content='너는 천문학과 관련된 지식을 답변하는 전문 에이전트이다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='태양계에서 가장 큰 행성은 무엇인가요?', additional_kwargs={}, response_metadata={})]

In [24]:
chain = prompt | llm | StrOutputParser()
print(chain.invoke({"user_input": "태양계에서 가장 큰 행성은 무엇인가요?"}))

태양계에서 가장 큰 행성은 **목성**입니다.

- **지름**: 약 14만 3천 km
- **특징**: 태양계에서 가장 무거운 가스 행성이며, 강한 자기장과 많은 위성을 가지고 있습니다.
- **대표적인 특징**: **대적점**이라고 불리는 거대한 폭풍이 있습니다.

원하시면 지구와 얼마나 큰지 비교해서도 설명해드릴게요.


# FewShotChatMessagePromptTemplate

In [25]:
examples4 = [
    {
        "instruction": "당신은 회의록 작성 전문가 입니다. 주어진 정보를 바탕으로 회의록을 작성해 주세요",
        "input": "2023년 12월 25일, XYZ 회사의 마케팅 전략 회의가 오후 3시에 시작되었다. 회의에는 마케팅 팀장인 김수진, 디지털 마케팅 담당자인 박지민, 소셜 미디어 관리자인 이준호가 참석했다. 회의의 주요 목적은 2024년 상반기 마케팅 전략을 수립하고, 새로운 소셜 미디어 캠페인에 대한 아이디어를 논의하는 것이었다. 팀장인 김수진은 최근 시장 동향에 대한 간략한 개요를 제공했으며, 이어서 각 팀원이 자신의 분야에서의 전략적 아이디어를 발표했다.",
        "answer": """
회의록: XYZ 회사 마케팅 전략 회의
일시: 2023년 12월 25일
장소: XYZ 회사 회의실
참석자: 김수진 (마케팅 팀장), 박지민 (디지털 마케팅 담당자), 이준호 (소셜 미디어 관리자)

1. 개회
   - 회의는 김수진 팀장의 개회사로 시작됨.
   - 회의의 목적은 2024년 상반기 마케팅 전략 수립 및 새로운 소셜 미디어 캠페인 아이디어 논의.

2. 시장 동향 개요 (김수진)
   - 김수진 팀장은 최근 시장 동향에 대한 분석을 제시.
   - 소비자 행동 변화와 경쟁사 전략에 대한 통찰 공유.

3. 디지털 마케팅 전략 (박지민)
   - 박지민은 디지털 마케팅 전략에 대해 발표.
   - 온라인 광고와 SEO 최적화 방안에 중점을 둠.

4. 소셜 미디어 캠페인 (이준호)
   - 이준호는 새로운 소셜 미디어 캠페인에 대한 아이디어를 제안.
   - 인플루언서 마케팅과 콘텐츠 전략에 대한 계획을 설명함.

5. 종합 논의
   - 팀원들 간의 아이디어 공유 및 토론.
   - 각 전략에 대한 예산 및 자원 배분에 대해 논의.

6. 마무리
   - 다음 회의 날짜 및 시간 확정.
   - 회의록 정리 및 배포는 박지민 담당.
"""
    },
    {
        "instruction": "당신은 요약 전문가 입니다. 다음 주어진 정보를 바탕으로 내용을 요약해 주세요",
        "input": "이 문서는 '지속 가능한 도시 개발을 위한 전략'에 대한 20페이지 분량의 보고서입니다. 보고서는 지속 가능한 도시 개발의 중요성, 현재 도시화의 문제점, 그리고 도시 개발을 지속 가능하게 만들기 위한 다양한 전략을 포괄적으로 다루고 있습니다. 이 보고서는 또한 성공적인 지속 가능한 도시 개발 사례를 여러 국가에서 소개하고, 이러한 사례들을 통해 얻은 교훈을 요약하고 있습니다.",
        "answer": """
문서 요약: 지속 가능한 도시 개발을 위한 전략 보고서

- 중요성: 지속 가능한 도시 개발이 필수적인 이유와 그에 따른 사회적, 경제적, 환경적 이익을 강조.
- 현 문제점: 현재의 도시화 과정에서 발생하는 주요 문제점들, 예를 들어 환경 오염, 자원 고갈, 불평등 증가 등을 분석.
- 전략: 지속 가능한 도시 개발을 달성하기 위한 다양한 전략 제시. 이에는 친환경 건축, 대중교통 개선, 에너지 효율성 증대, 지역사회 참여 강화 등이 포함됨.
- 사례 연구: 전 세계 여러 도시의 성공적인 지속 가능한 개발 사례를 소개. 예를 들어, 덴마크의 코펜하겐, 일본의 요코하마 등의 사례를 통해 실현 가능한 전략들을 설명.
- 교훈: 이러한 사례들에서 얻은 주요 교훈을 요약. 강조된 교훈에는 다각적 접근의 중요성, 지역사회와의 협력, 장기적 계획의 필요성 등이 포함됨.

이 보고서는 지속 가능한 도시 개발이 어떻게 현실적이고 효과적인 형태로 이루어질 수 있는지에 대한 심도 있는 분석을 제공합니다.
"""
    }
]

In [26]:
question = {
    "instruction": "회의록을 작성해 주세요",
    "input": "2023년 12월 26일, ABC 기술 회사의 제품 개발 팀은 새로운 모바일 애플리케이션 프로젝트에 대한 주간 진행 상황 회의를 가졌다. 이 회의에는 프로젝트 매니저인 최현수, 주요 개발자인 황지연, UI/UX 디자이너인 김태영이 참석했다. 회의의 주요 목적은 프로젝트의 현재 진행 상황을 검토하고, 다가오는 마일스톤에 대한 계획을 수립하는 것이었다. 각 팀원은 자신의 작업 영역에 대한 업데이트를 제공했고, 팀은 다음 주까지의 목표를 설정했다.",
}

In [30]:
# 모델 생성, 스트리밍 기능 추가
llm = init_chat_model(model="gpt-5.4",
                      temperature=0, max_tokens=500,
                      streaming = True,
                      callbacks = [StreamingStdOutCallbackHandler()],
                      )

# (1) 예시 선택기 설정
example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples4, OpenAIEmbeddings(), FAISS, k=1
)

# (2)메세지 프롬프트 생성
message = ChatPromptTemplate.from_messages(
    [
        ("system", "{instruction}"),
        ("human", "{input}"),
        ("ai", "{answer}"),
    ]
)

# (3) 선택된 예시 1개로 프롬프트를 생성
fewshot_pormpt = FewShotChatMessagePromptTemplate(
    example_selector=example_selector,
    example_prompt=message,
)

# (4) 사용자 질문이 포함된 최종프롬프트를 생성
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "{instruction}"),
        fewshot_pormpt,
        ("human", "{input}")
    ]
)

# (5) 체인 생성 및 실행
chain = prompt | llm | StrOutputParser()

answer = chain.invoke(question)

회의록: ABC 기술 회사 제품 개발 주간 진행 회의

- **일시:** 2023년 12월 26일  
- **주제:** 새로운 모바일 애플리케이션 프로젝트 주간 진행 상황 점검  
- **참석자:** 최현수 (프로젝트 매니저), 황지연 (주요 개발자), 김태영 (UI/UX 디자이너)

### 1. 회의 목적
- 프로젝트의 현재 진행 상황을 검토
- 다가오는 마일스톤에 대한 계획 수립
- 다음 주까지의 목표 설정

### 2. 진행 상황 공유
- **최현수 (프로젝트 매니저)**
  - 전체 프로젝트 진행 현황을 점검
  - 주요 일정 및 마일스톤 계획을 공유

- **황지연 (주요 개발자)**
  - 개발 진행 상태에 대한 업데이트 제공
  - 현재 구현 중인 기능 및 향후 개발 일정 설명

- **김태영 (UI/UX 디자이너)**
  - 디자인 작업 현황을 공유
  - 사용자 인터페이스 및 사용자 경험 관련 진행 사항 설명

### 3. 주요 논의 사항
- 현재 프로젝트 진행 상태 전반 검토
- 다가오는 마일스톤 달성을 위한 준비 사항 논의
- 각 담당 영역별 협업 필요 사항 확인

### 4. 다음 주 목표
- 팀원별 작업 목표를 구체화
- 다음 주까지 완료해야 할 주요 업무 설정
- 마일스톤 일정에 맞춘 우선순위 재정리

### 5. 결론
- 각 팀원은 자신의 담당 업무를 지속적으로 수행하기로 함
- 다음 주 목표를 기준으로 진행 상황을 점검하기로 함
- 차기 회의에서 마일스톤 준비 상태를 다시 확인하기로 함

필요하시면 제가 이 회의록을  
1) **더 공식적인 문서 형식**,  
2) **간단한 요약형**,  
3) **액션 아이템 중심 형식**  
중 하나로 다시 작성해드릴 수 있습니다.

In [32]:
question = {
    "instruction": "너는 문서요약 전문가이다. 입력되는 문장을 요약한다",
    "input": "이 문서는 '양자컴퓨터 개발을 위한 전략'에 대한 50페이지 분량의 보고서입니다. 보고서는 양자 컴퓨터 개발의 중요성, 현재 컴퓨터 시스템의 문제점, 그리고 양자 컴퓨터 개발을 지속 가능하게 만들기 위한 다양한 전략을 포괄적으로 다루고 있습니다. 이 보고서는 또한 성공적인 지속 가능한 양자 컴퓨터 개발 사례를 여러 국가에서 소개하고, 이러한 사례들을 통해 얻은 교훈을 요약하고 있습니다."}


In [33]:
llm = init_chat_model(model="gpt-5.4",
                      temperature=0, max_tokens=500,
                      streaming = True,
                      callbacks = [StreamingStdOutCallbackHandler()],
                      )

example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples4, OpenAIEmbeddings(), FAISS, k=1
)

message = ChatPromptTemplate.from_messages(
    [
        ("system", "{instruction}"),
        ("human", "{input}"),
        ("ai", "{answer}"),
    ]
)

fewshot_pormpt = FewShotChatMessagePromptTemplate(
    example_selector=example_selector,
    example_prompt=message,
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "{instruction}"),
        fewshot_prompt,
        ("human", "{input}")
    ]
)

chain = prompt | llm | StrOutputParser()

answer = chain.invoke(question)

문서 요약: 양자컴퓨터 개발을 위한 전략 보고서

- 중요성: 양자컴퓨터 개발이 미래 산업과 과학기술 경쟁력 확보에 핵심적이며, 복잡한 계산 문제 해결, 신약 개발, 암호기술, 최적화 분야 등에서 큰 잠재력을 가진다는 점을 강조합니다.
- 현재 문제점: 기존 컴퓨터 시스템이 처리 속도와 계산 복잡성 측면에서 한계를 가지며, 초고속 연산이 필요한 문제 해결에 제약이 있다는 점을 분석합니다. 또한 양자컴퓨터 개발 과정에서 높은 비용, 기술적 난제, 전문 인력 부족 등의 문제도 다룹니다.
- 전략: 양자컴퓨터 개발을 지속 가능하게 추진하기 위한 다양한 전략을 제시합니다. 여기에는 핵심 기술 연구개발 투자 확대, 산학연 협력 강화, 전문 인력 양성, 안정적인 부품 및 인프라 구축, 국제 협력 확대 등이 포함됩니다.
- 사례 연구: 여러 국가의 성공적인 양자컴퓨터 개발 사례를 소개하며, 각국이 연구개발 지원, 국가 차원의 전략 수립, 민간 기업과 연구기관의 협력을 통해 성과를 이루어낸 과정을 설명합니다.
- 교훈: 사례들을 통해 얻은 주요 교훈으로는 장기적이고 일관된 투자, 국가 차원의 체계적 지원, 협력 생태계 조성, 기술 개발과 인재 육성의 병행 필요성 등이 강조됩니다.

이 보고서는 양자컴퓨터 개발의 필요성과 현실적 과제를 균형 있게 다루면서, 지속 가능하고 효과적인 발전 방향을 제시하는 종합적인 분석 자료입니다.

# Partial Prompt

## 문자열 값을 사용한 부분 포맷팅

In [35]:
prompt = PromptTemplate.from_template("지구의 {layer}에서 가장 흔한 원소는 {element}입니다.")

partial_prompt = prompt.partial(layer="대기")

print(partial_prompt.format(element="산소"))

지구의 대기에서 가장 흔한 원소는 산소입니다.


- 프롬프트 초기화 시에 부분 변수를 지정 가능

In [36]:
prompt = PromptTemplate(
    template = "지구의 {layer}에서 가장 흔한 원소는 {element}입니다.",
    input_variables = ["element"],
    partial_variables = {"layer" : "맨틀"}
)

print(partial_prompt.format(element="규소"))

지구의 대기에서 가장 흔한 원소는 규소입니다.


## 문자열 값을 반환하는 함수를 사용한 부분 포맷팅
- 동적으로 값을 생성해야 할 때

In [40]:
# 현재 날짜를 기반으로 계절을 자동으로 할당 함수
from datetime import datetime

def get_current_season():
    month = datetime.now().month

    if 3 <= month <= 5 :
        return "봄"
    elif 6<= month <= 8 :
        return "여름"
    elif 9<= month <= 11 :
        return "가을"
    elif 12 <= month <= 2 :
        return "겨울"

# 함수를 이용해서 season 변수를 자동을 할당
prompt = PromptTemplate(
    template="지구의 {season}에서 일어나는 대표적인 {event}에 대해 알려줘",
    #사용자가 직접 할당
    input_variables=["event"],
    #함수를 이용해서 자동으로 할당
    partial_variables={"season": get_current_season}
)

print(prompt.format(event="자연 현상"))

지구의 여름에서 일어나는 대표적인 자연 현상에 대해 알려줘


In [41]:
chain = prompt | llm | StrOutputParser()

chain.invoke({"event": "자연 현상"})

지구의 여름에는 기온 상승과 일조량 증가 때문에 여러 대표적인 자연 현상이 나타납니다. 대표적인 것들을 간단히 정리하면 다음과 같습니다.

1. **기온 상승**
- 태양 고도가 높아지고 낮이 길어져 지표면이 더 많이 가열됩니다.
- 이 때문에 전반적으로 더운 날씨가 이어집니다.

2. **장마와 집중호우**
- 특히 동아시아 지역에서는 여름철에 장마가 나타납니다.
- 따뜻하고 습한 공기가 유입되면서 비가 오래 내리거나 짧은 시간에 매우 많은 비가 내리기도 합니다.

3. **태풍 발생**
- 여름과 초가을에는 바닷물 온도가 높아져 태풍이 발생하기 쉬워집니다.
- 강한 바람과 많은 비를 동반해 큰 피해를 줄 수 있습니다.

4. **천둥번개와 소나기**
- 강한 햇빛으로 지표면이 뜨거워지면 공기가 빠르게 상승합니다.
- 이 과정에서 적란운이 발달해 소나기, 천둥, 번개가 자주 발생합니다.

5. **식물의 왕성한 성장**
- 햇빛과 비가 충분해 식물이 빠르게 자랍니다.
- 숲과 들판이 더욱 푸르게 변하고 농작물도 활발히 자랍니다.

6. **곤충과 동물의 활동 증가**
- 더운 날씨 속에서 모기, 매미 같은 곤충의 활동이 활발해집니다.
- 많은 동물들도 번식과 먹이 활동이 왕성해집니다.

7. **해양 현상 변화**
- 바다 표면 온도가 올라가면서 해무, 적조 같은 현상이 나타날 수 있습니다.
- 일부 지역에서는 산호 백화 현상도 심해질 수 있습니다.

8. **폭염과 열대야**
- 여름철에는 매우 더운 날씨가 지속되는 폭염이 발생할 수 있습니다.
- 밤에도 기온이 충분히 내려가지 않는 열대야 현상도 흔합니다.

원하시면 제가 이것을 **초등학생용으로 쉽게**, 또는 **학교 과제용으로 자세히** 다시 정리해드릴 수

'지구의 여름에는 기온 상승과 일조량 증가 때문에 여러 대표적인 자연 현상이 나타납니다. 대표적인 것들을 간단히 정리하면 다음과 같습니다.\n\n1. **기온 상승**\n- 태양 고도가 높아지고 낮이 길어져 지표면이 더 많이 가열됩니다.\n- 이 때문에 전반적으로 더운 날씨가 이어집니다.\n\n2. **장마와 집중호우**\n- 특히 동아시아 지역에서는 여름철에 장마가 나타납니다.\n- 따뜻하고 습한 공기가 유입되면서 비가 오래 내리거나 짧은 시간에 매우 많은 비가 내리기도 합니다.\n\n3. **태풍 발생**\n- 여름과 초가을에는 바닷물 온도가 높아져 태풍이 발생하기 쉬워집니다.\n- 강한 바람과 많은 비를 동반해 큰 피해를 줄 수 있습니다.\n\n4. **천둥번개와 소나기**\n- 강한 햇빛으로 지표면이 뜨거워지면 공기가 빠르게 상승합니다.\n- 이 과정에서 적란운이 발달해 소나기, 천둥, 번개가 자주 발생합니다.\n\n5. **식물의 왕성한 성장**\n- 햇빛과 비가 충분해 식물이 빠르게 자랍니다.\n- 숲과 들판이 더욱 푸르게 변하고 농작물도 활발히 자랍니다.\n\n6. **곤충과 동물의 활동 증가**\n- 더운 날씨 속에서 모기, 매미 같은 곤충의 활동이 활발해집니다.\n- 많은 동물들도 번식과 먹이 활동이 왕성해집니다.\n\n7. **해양 현상 변화**\n- 바다 표면 온도가 올라가면서 해무, 적조 같은 현상이 나타날 수 있습니다.\n- 일부 지역에서는 산호 백화 현상도 심해질 수 있습니다.\n\n8. **폭염과 열대야**\n- 여름철에는 매우 더운 날씨가 지속되는 폭염이 발생할 수 있습니다.\n- 밤에도 기온이 충분히 내려가지 않는 열대야 현상도 흔합니다.\n\n원하시면 제가 이것을 **초등학생용으로 쉽게**, 또는 **학교 과제용으로 자세히** 다시 정리해드릴 수'

# 여러 개의 프롬프트 연결하기

In [42]:
prompt1 = PromptTemplate.from_template("{area1}과 {area2} 간의 거리를 알려줘")
prompt2 = "\n\n{transport}로 얼마나 걸리는지 알려줘."
prompt3 = "\n\n{language}로 번역해줘"
prompt4 = PromptTemplate.from_template("\n\n{num}자 이내로 요약해줘.")

combined_prompt = prompt1 + prompt2 + prompt3 + prompt4

combined_prompt.format(area1="서울", area2="부산", transport="기차", language="영어", num=20)

'서울과 부산 간의 거리를 알려줘\n\n기차로 얼마나 걸리는지 알려줘.\n\n영어로 번역해줘\n\n20자 이내로 요약해줘.'

In [49]:
llm = init_chat_model(model="gpt-5.4", temperature=0, max_tokens=500)

chain = combined_prompt | llm | StrOutputParser()

answer = chain.invoke({"area1" : "서울",
                       "area2" : "부산",
                       "transport" : "기차",
                       "language" : "영어",
                       "num" : 20})

display(Markdown(answer))

서울과 부산은 약 325km 정도 떨어져 있어요.  
기차로는 KTX 기준 약 2시간 15분~2시간 40분 걸려요.  

영어 번역:  
"The distance between Seoul and Busan is about 325 km. By train, it takes about 2 hours 15 minutes to 2 hours 40 minutes."

20자 이내 요약:  
서울-부산 325km, 2시간대

# Multi Chain

## 순차적인 체인 연결

In [50]:
llm = init_chat_model(model="gpt-5.4", temperature=0, max_tokens=1000)

prompt1 = PromptTemplate.from_template("{kr_word}를 영어로 번역해줘")

chain1 = prompt1 | llm | StrOutputParser()

chain1.invoke({"kr_word":"미래"})

'"미래"는 영어로 **future** 입니다.'

In [51]:
prompt2 = PromptTemplate.from_template("""
    {eng_word}의 뜻을 옥스퍼드 사전을 사용해서 영어로 알려줘""")

chain2 = {"eng_word" : chain1} | prompt2 | llm | StrOutputParser()

chain2.invoke({"kr_word":"미래"})

'According to the Oxford Learner’s Dictionaries, **future** means:\n\n**“the period of time that will come after the present or the events that will happen then.”**\n\nExample:\n- **Nobody knows what will happen in the future.**\n\n원하시면 제가 이 단어의 **쉬운 뜻**, **발음**, **예문**, **future와 futuristic의 차이**도 함께 설명해드릴게요.'

요약이나 번역 체인을 만들어서 추가하기

In [60]:
prompt3 = PromptTemplate.from_template("""
    {eng_means}의 뜻을 120자 이내로 요약해서 한글로 번역해줘 """)

chain3 =  {"eng_means" : chain2} | prompt3 | llm | StrOutputParser()

chain3.invoke({"kr_word":"미래"})

'future: ① 현재 이후의 시간, 앞으로 일어날 일 ② 앞으로 성공하거나 계속 존재할 가능성'

## 다단계 순차 체인

In [61]:
llm = init_chat_model(model="gpt-4o-mini", temperature=0, max_tokens=1000)

# 1단계 : 분석을 위한 프롬프트
analyze_prompt = ChatPromptTemplate.from_template(
    "다음 주제와 관련된 핵심 키워드를 3개만 추출해줘 : {topic}"
)

# 2단계 : 개요 작성을 위한 프롬프트
outline_prompt = ChatPromptTemplate.from_template(
    "다음 키워드를 기반으로 글의 개요을 작성해줘 : {keyword}"
)

# 3단계 : 본문 작성을 위한 프롬프트
content_prompt = ChatPromptTemplate.from_template(
    "다음 개요를 기반으로 글의 본문을 작성해줘 : {outline}"
)

chain = (
    # invoke()에서 설정한 파라미터 값을 가져와서 할당
    {"topic": RunnablePassthrough()} |
    # analyze_prompt의 결과를 outline_prompt의 keyword에 할당해서 실행
    RunnablePassthrough.assign(keyword=analyze_prompt | llm | StrOutputParser()) |
    # outline_prompt의 결과를 content_prompt의 outline에 할당해서 실행
    RunnablePassthrough.assign(outline=outline_prompt | llm | StrOutputParser()) |
    # content_prompt를 실행
    content_prompt | llm | StrOutputParser()
)

answer = chain.invoke("양자컴퓨터의 전망과 미래")

display(Markdown(answer))

### 양자우위와 산업 혁신

#### I. 서론
양자우위(Quantum Advantage)는 양자 컴퓨터가 고전 컴퓨터에 비해 특정 문제를 더 빠르고 효율적으로 해결할 수 있는 능력을 의미합니다. 이는 단순한 기술적 진보를 넘어, 산업 전반에 걸쳐 혁신을 촉진할 수 있는 잠재력을 지니고 있습니다. 양자 컴퓨팅의 발전은 20세기 후반부터 시작되었으며, 최근 몇 년간 급속한 기술적 진전을 이루어냈습니다. 이러한 발전은 데이터 처리, 암호화, 최적화 문제 해결 등 다양한 분야에서 새로운 가능성을 열어주고 있습니다. 본 글에서는 양자우위의 개념과 이를 실현하기 위한 기술적 도전 과제, 그리고 양자 알고리즘이 산업 혁신에 미치는 영향을 살펴보겠습니다.

#### II. 양자우위(Quantum Advantage)
양자우위는 양자 컴퓨터가 고전 컴퓨터에 비해 특정 문제를 해결하는 데 있어 우위를 점하는 상황을 설명합니다. 예를 들어, 고전 컴퓨터는 대규모 소인수 분해 문제를 해결하는 데 수천 년이 걸릴 수 있지만, 쇼어 알고리즘을 활용한 양자 컴퓨터는 이를 몇 초 만에 해결할 수 있습니다. 이러한 우위는 양자 비트(큐비트)의 특성, 즉 중첩과 얽힘을 통해 가능해집니다. 그러나 양자우위를 실현하기 위해서는 여러 기술적 도전 과제가 존재합니다. 특히, 오류 수정 및 큐비트의 안정성 문제는 양자 컴퓨터의 실용화를 위한 주요 장애물로 남아 있습니다.

#### III. 양자 알고리즘(Quantum Algorithms)
양자 알고리즘은 양자 컴퓨터의 성능을 극대화하는 핵심 요소입니다. 대표적인 양자 알고리즘으로는 쇼어 알고리즘과 그로버 알고리즘이 있습니다. 쇼어 알고리즘은 소인수 분해 문제를 효율적으로 해결할 수 있는 방법을 제공하며, 이는 현대 암호 시스템의 기반을 흔들 수 있는 잠재력을 지니고 있습니다. 반면, 그로버 알고리즘은 비구조적 데이터베이스에서의 검색 속도를 획기적으로 향상시킵니다. 이러한 양자 알고리즘은 데이터 분석, 암호 해독, 최적화 문제 해결 등 다양한 산업 분야에서 응용될 수 있으며, 이는 기업의 경쟁력을 크게 향상시킬 수 있습니다.

#### IV. 산업 혁신(Industrial Innovation)
양자 컴퓨팅은 제조업, 금융, 의료 등 다양한 산업에 혁신적인 변화를 가져오고 있습니다. 예를 들어, 제조업에서는 양자 컴퓨터를 활용한 최적화 알고리즘이 생산 공정을 개선하고 비용을 절감하는 데 기여할 수 있습니다. 금융 분야에서는 양자 알고리즘을 통해 리스크 분석 및 포트폴리오 최적화가 가능해지며, 이는 투자 전략의 효율성을 높이는 데 도움을 줍니다. 의료 분야에서도 양자 컴퓨팅은 신약 개발 및 유전자 분석에 혁신적인 기여를 할 것으로 기대됩니다. 이러한 양자 기술을 활용한 새로운 비즈니스 모델은 기업의 경쟁력을 강화하고, 시장에서의 우위를 점하는 데 중요한 역할을 할 것입니다.

#### V. 결론
양자우위와 양자 알고리즘은 산업 혁신에 기여하는 중요한 요소로 자리 잡고 있습니다. 양자 컴퓨팅의 발전은 다양한 산업 분야에서의 문제 해결 방식을 혁신적으로 변화시킬 것으로 기대됩니다. 향후 양자 컴퓨팅의 발전 방향은 더욱 가속화될 것이며, 이는 기업과 연구 기관 간의 협력이 필수적임을 의미합니다. 이러한 협력을 통해 양자 기술의 상용화가 이루어지고, 산업 전반에 걸쳐 긍정적인 변화가 일어날 것입니다.

#### VI. 참고 문헌
- 관련 연구 및 자료 목록
- 양자 컴퓨팅 관련 서적 및 논문
- 산업 혁신 사례 연구 자료

## 병렬 체인 실행

In [62]:
llm = init_chat_model(model="gpt-5.4", temperature=0, max_tokens=1000)

positive_prompt = ChatPromptTemplate.from_template(
    "{topic}의 긍정적인 측면에 대해 3가지만 설명해줘"
)

negative_prompt = ChatPromptTemplate.from_template(
    "{topic}의 부정적인 측면에 대해 3가지만 설명해줘"
)

netural_prompt = ChatPromptTemplate.from_template(
    "{topic}의 중립적인(객관적인) 측면에 대해 3가지만 설명해줘"
)

#병렬 체인 구성
chain = RunnableParallel(
    positive = positive_prompt | llm | StrOutputParser(),
    negative = negative_prompt | llm | StrOutputParser(),
    netural = netural_prompt | llm | StrOutputParser(),
)

answer = chain.invoke("의과대학의 정원증가정책")

print("===긍정적인 측면===")
display(Markdown(answer["positive"]))
print("===부정적인 측면===")
display(Markdown(answer["negative"]))
print("===중립적인 측면===")
display(Markdown(answer["netural"]))

===긍정적인 측면===


의과대학 정원 증가 정책의 **긍정적인 측면**은 크게 다음 3가지로 설명할 수 있습니다.

1. **의료 인력 부족 문제 완화**  
   의과대학 입학 정원이 늘어나면 장기적으로 의사 수가 증가하게 됩니다. 이는 현재 부족한 의료 인력을 보충하는 데 도움이 되며, 특히 고령화로 인해 의료 수요가 늘어나는 사회에서 중요한 의미가 있습니다.

2. **지역 의료 격차 해소 가능성**  
   수도권에 비해 지방이나 농어촌 지역은 의사 수가 부족한 경우가 많습니다. 정원 확대와 함께 지역의사 양성 정책이 병행되면, 의료 취약 지역에 더 많은 의사를 배치할 수 있어 지역 간 의료 서비스 격차를 줄이는 데 기여할 수 있습니다.

3. **국민의 의료 접근성 향상**  
   의사 수가 늘어나면 진료 대기 시간이 줄어들고, 보다 많은 국민이 적절한 시기에 의료 서비스를 받을 가능성이 높아집니다. 이는 국민 건강 증진과 의료 서비스의 전반적인 질 향상에도 긍정적인 영향을 줄 수 있습니다.

원하시면 제가 이어서 **부정적인 측면 3가지**도 같이 정리해드릴게요.

===부정적인 측면===


의과대학 정원 증가 정책의 부정적인 측면은 다음 3가지로 설명할 수 있습니다.

1. **교육의 질 저하 가능성**  
   의대생 수가 갑자기 늘어나면 교수진, 실습 병원, 해부실, 실험실 등 교육 인프라가 충분히 확충되지 못할 수 있습니다. 그 결과 학생 1인당 받을 수 있는 교육과 임상실습의 질이 낮아질 가능성이 있습니다.

2. **의료 인력 불균형 문제의 근본 해결 한계**  
   단순히 의사 수만 늘린다고 해서 지역 간, 진료과 간 불균형이 자동으로 해결되지는 않습니다. 예를 들어, 지방이나 필수의료 분야는 여전히 기피될 수 있어서, 실제로 필요한 곳에 의사가 배치되지 않을 가능성이 있습니다.

3. **기존 의료체계와의 갈등 및 사회적 비용 증가**  
   정원 확대는 의사단체, 정부, 교육기관 사이의 갈등을 유발할 수 있고, 이 과정에서 사회적 비용이 커질 수 있습니다. 또한 추가 학생 교육을 위한 예산, 시설 투자, 수련 환경 개선 비용도 많이 들어갑니다.

원하시면 제가 이것을 **찬반토론용 형식**이나 **시험 답안형 문장**으로도 바꿔드릴 수 있습니다.

===중립적인 측면===


의과대학 정원 증가 정책을 **중립적·객관적 관점**에서 보면, 주로 다음 3가지 측면에서 설명할 수 있습니다.

1. **의료 인력 수급 조절 기능**
   - 의대 정원 증가는 장기적으로 의사 수를 늘려 의료 인력 부족 문제를 완화하려는 정책 수단입니다.
   - 특히 고령화, 만성질환 증가, 지역 간 의료 격차 같은 변화에 대응하기 위해 활용됩니다.
   - 다만 실제 현장에 의사가 배출되기까지는 긴 시간이 걸리므로, 효과는 단기보다 중장기적으로 나타납니다.

2. **지역·전문과목 불균형 문제와의 연관성**
   - 정원 확대는 단순히 전체 의사 수를 늘리는 효과는 있지만, 지역 의료나 필수의료 분야 부족 문제를 자동으로 해결하는 것은 아닙니다.
   - 예를 들어 수도권 집중이나 특정 인기 진료과 쏠림 현상이 계속되면, 정원 증가만으로는 정책 목표가 충분히 달성되지 않을 수 있습니다.
   - 따라서 객관적으로 보면 정원 증가는 “양적 확대” 정책이며, “배치와 분포” 문제는 별도 대책이 필요합니다.

3. **교육·수련 체계에 미치는 영향**
   - 의대 정원이 늘어나면 대학의 교육 인프라, 교수 인력, 실습 병원, 전공의 수련 환경도 함께 확충되어야 합니다.
   - 이런 기반이 충분하면 의료인력 확대 효과를 기대할 수 있지만, 준비가 부족하면 교육의 질 관리가 중요한 쟁점이 됩니다.
   - 즉, 정원 증가는 단순 입학 인원 조정이 아니라 의료교육 시스템 전반과 연결된 정책입니다.

원하시면 제가 이것을 **찬반 의견 없이 더 짧게 요약**하거나, **발표용 문장 형태**로 바꿔드릴 수 있습니다.

## 병렬 결과 통합

In [63]:
summary_prompt = ChatPromptTemplate.from_template(
    """
    다음 내용을 3가지 측면을 기반으로 종합적인 결론을 내려줘 :

    - 장점 : {positive}
    - 단점 : {negative}
    - 주의사항 : {netural}
    """
)

chain2 = (chain | summary_prompt | llm | StrOutputParser())

answer = chain2.invoke("양자컴퓨터의 전망과 미래")

display(Markdown(answer))

종합적으로 보면, **양자컴퓨터의 미래는 매우 유망하지만 과도한 낙관도, 지나친 비관도 경계해야 하는 분야**라고 결론내릴 수 있습니다.

### 1. 장점 측면
양자컴퓨터는 신약 개발, 신소재 설계, 최적화, 인공지능 등에서 **기존 컴퓨터가 풀기 어려운 문제를 해결할 가능성**이 있습니다. 또한 관련 연구가 양자역학, 반도체, 통신, 암호기술 등 **다른 과학기술 발전까지 촉진**한다는 점에서 미래 산업의 중요한 성장 동력으로 평가됩니다.

### 2. 단점 측면
반면, 양자컴퓨터가 충분히 발전할 경우 **기존 암호체계가 위협받아 보안 문제가 커질 수 있고**, 기술 개발 비용이 매우 높아 **국가·기업 간 기술 독점과 격차 심화** 가능성도 있습니다. 또한 아직은 오류 보정, 큐비트 안정성, 운영 비용 등 **실용화의 장벽이 매우 큰 기술**이기도 합니다.

### 3. 주의사항 측면
객관적으로 볼 때 양자컴퓨터는 **모든 분야를 바꾸는 만능 기술이 아니라 특정 문제에 강한 특수 목적 기술**에 가깝습니다. 따라서 기존 컴퓨터를 완전히 대체하기보다는 **보완적으로 활용될 가능성**이 크며, 사회적 영향도 단번에 나타나기보다 **점진적으로 확대될 가능성**이 높습니다.

## 최종 결론
따라서 양자컴퓨터의 미래는 **“장기적으로 큰 잠재력이 있지만, 현실적 한계와 부작용을 함께 관리해야 하는 기술”**이라고 볼 수 있습니다.  
즉, **기대할 가치는 충분하지만, 상용화 속도·보안 대비·기술 격차 문제를 신중하게 준비하면서 접근해야 한다**는 것이 가장 균형 잡힌 결론입니다. 

원하시면 이 내용을 **발표용 결론 문장**이나 **보고서형 요약**으로도 바꿔드릴게요.

## 함수 기반 라우팅

In [65]:
#주어진 픽에 따라 라우팅하는 함수

def route_by_topic(input_dic) :
    """ 주제에 따른 다른 체인을 선택하는 기능 """
    #딕셔너리에서 key도 value를 가져오는 방법
    #   -딕셔너리명["key"]
    #   -딕셔너리명.get("key", "초기값")
    topic = input_dic.get("topic", "").lower()

    if "code" in topic or "programming" in topic :
        return "technical"
    elif "business" in topic or "finance" in topic :
        return "business"
    else :
        return "general"

chains = {
    "technical" : ChatPromptTemplate.from_template(
        "기술 전문가 입장에서 답변 : {question}") | llm | StrOutputParser(),
    "business" : ChatPromptTemplate.from_template(
        "경영 전문가 입장에서 답변 : {question}") | llm | StrOutputParser(),
    "general" : ChatPromptTemplate.from_template(
        "일반인 입장에서 답변 : {question}") | llm | StrOutputParser(),
}

# 체인 라우팅 함수
def route_chain(input_dic) :
    topic = route_by_topic(input_dic)
    return chains[topic].invoke(input_dic)

# 함수 -> chain
router_chain = RunnableLambda(route_chain)

answer = router_chain.invoke(
    {"topic" : "Python Programming",
     "question" : "효율적인 코딩 작성 방법"}
)

display(Markdown(answer))

효율적인 코딩 작성 방법을 **기술 전문가 관점**에서 정리하면, 핵심은 단순히 “빨리 짜는 것”이 아니라 **읽기 쉽고, 유지보수 가능하며, 성능과 품질까지 고려한 코드**를 작성하는 것입니다.  
아래 원칙들이 가장 중요합니다.

---

## 1. 먼저 설계하고 코딩하기
효율적인 코딩은 키보드보다 **문제 정의와 설계**에서 시작됩니다.

### 체크할 것
- **무엇을 해결할 것인지** 명확히 정의
- 입력값 / 출력값 / 예외 상황 정리
- 성능 요구사항 확인
- 확장 가능성 고려

### 좋은 방법
- 큰 문제를 **작은 단위의 기능**으로 나누기
- 흐름도를 간단히 그리기
- 의사코드(pseudocode) 먼저 작성

예:
```text
1. 사용자 입력 받기
2. 입력값 검증
3. 데이터 처리
4. 결과 출력
5. 오류 발생 시 예외 처리
```

설계 없이 바로 코딩하면 수정 비용이 커집니다.

---

## 2. 가독성이 좋은 코드 작성
전문가들은 “잘 동작하는 코드”보다 **남이 읽을 수 있는 코드**를 더 높게 평가합니다.

### 핵심 원칙
- 변수명은 의미 있게 작성
- 함수명은 동작이 드러나게 작성
- 한 함수는 **하나의 책임**만 가지게 작성
- 복잡한 조건문은 분리

### 나쁜 예
```python
x = 10
y = 20
z = x * y
```

### 좋은 예
```python
width = 10
height = 20
area = width * height
```

코드는 미래의 나 또는 다른 개발자가 읽는 문서입니다.

---

## 3. 중복을 줄이기 (DRY 원칙)
DRY(Don’t Repeat Yourself)는 효율적인 코딩의 핵심입니다.

### 비효율적 예
```python
print("홍길동님의 주문 금액은 10000원입니다.")
print("김철수님의 주문 금액은 15000원입니다.")
print("이영희님의 주문 금액은 12000원입니다.")
```

### 효율적 예
```python
def print_order(customer_name, amount):
    print(f"{customer_name}님의 주문 금액은 {amount}원입니다.")
```

중복이 많으면 수정 시 여러 곳을 동시에 바꿔야 하므로 버그 가능성이 증가합니다.

---

## 4. 함수와 모듈로 분리하기
코드를 작은 단위로 나누면 재사용성과 테스트 편의성이 올라갑니다.

### 좋은 구조 예
```python
def validate_input(data):
    pass

def process_data(data):
    pass

def save_result(result):
    pass
```

### 장점
- 디버깅 쉬움
- 재사용 가능
- 협업에 유리
- 테스트 작성 쉬움

---

## 5. 예외 처리와 방어적 코딩
효율적인 코드는 정상 동작만 고려하지 않습니다.  
**잘못된 입력, 네트워크 오류, 파일 누락, null 값** 같은 상황도 처리해야 합니다.

### 예
```python
try:
    number = int(input("숫자 입력: "))
    print(10 / number)
except ValueError:
    print("숫자를 입력해야 합니다.")
except ZeroDivisionError:
    print("0으로 나눌 수 없습니다.")
```

### 전문가 관점 포인트
- 에러를 숨기지 말고 명확히 처리
- 예외 메시지는 원인 파악이 가능해야 함
- 로그를 남겨 운영 이슈를 추적 가능하게 하기

---

## 6. 자료구조와 알고리즘을 적절히 선택하기
효율성은 코드 스타일뿐 아니라 **시간 복잡도와 공간 복잡도**에도 달려 있습니다.

### 예시
- 검색이 많으면 `list`보다 `set` / `dict`
- FIFO 구조는 queue
- 정렬 빈도가 높으면 자료 구조 재검토

### 예
```python
# 비효율적
items = [1, 2, 3, 4, 5]
if 5 in items:
    print("존재")

# 더 효율적
items = {1, 2, 3, 4, 5}
if 5 in items:
    print("존재")
```

### 핵심
- 작은

# Agent기반 라우팅

In [74]:
def agent_route_by_topic(context) :
    """ AI가 내용을 분석해서 토픽을 선택하고 체인을 구성 """
    route_prompt = ChatPromptTemplate.from_template(
        """
        {context}를 분석해서 다음 주제에서 가장 적절한 주제를 선택해줘
        - **technical** : 기술과 관련된 질의인 경우에 선택
        - **business** : 경영/사업과 관련된 질의인 경우에 선택
        - **general** : 기타 질의인 경우에 선택

        오직 **3가지 중 해당 주제의 명칭만 반환**해줘
        """
    )

    chain = route_prompt | llm | StrOutputParser()

    return chain.invoke({"context" : context})

chains = {
    "technical" : ChatPromptTemplate.from_template(
        """기술부서 담당자 입장에서 답변 : {context}
        -> **기술 전문가**가 답변했다는 것을 표기해줘""") | llm | StrOutputParser(),
    "business" : ChatPromptTemplate.from_template(
        """영업부서 담당자 입장에서 답변 : {context}
        -> **영업 전문가**가 답변했다는 것을 표기해줘""") | llm | StrOutputParser(),
    "general" : ChatPromptTemplate.from_template(
        """민원부서 담당자 입장에서 답변 : {context}
        -> **민원부서 담당자**가 답변했다는 것을 표기해줘""") | llm | StrOutputParser(),
}


def agent_route_chain(context) :
    topic = agent_route_by_topic(context)

    return chains[topic].invoke(context)

agent_router_chain = RunnableLambda(agent_route_chain)

answer = agent_router_chain.invoke(
    {"context" : "귀사의 냉장고 사용설명서를 받을 수 있을까요"}
)

display(Markdown(answer))

안녕하세요, **민원부서 담당자**입니다.

문의하신 **귀사 냉장고 사용설명서 제공** 관련하여 안내드립니다.

사용설명서는 **제품 모델명 확인 후 제공이 가능**합니다.  
정확한 안내를 위해 **냉장고 모델명** 또는 **제품 사진(모델명이 표시된 라벨 포함)**을 보내주시면 확인 후 사용설명서를 안내해드리겠습니다.

추가로, 제품에 따라 **공식 홈페이지에서 사용설명서를 다운로드**하실 수 있는 경우도 있으니, 모델명 확인 후 함께 안내드리겠습니다.

감사합니다.

## 다중 모델 앙상블

In [76]:
gpt_model1 = init_chat_model(model="gpt-4o")
gpt_model2 = init_chat_model(model="gpt-5.5")
gemini_model = init_chat_model(model="google_genai:gemini-2.5-flash")

prompt = ChatPromptTemplate.from_template("{question}")

chain = RunnableParallel(
    gpt = prompt | gpt_model1 | StrOutputParser(),
    gemini = prompt | gemini_model | StrOutputParser(),
)

final_prompt = ChatPromptTemplate.from_template(
    """
    두 AI의 응답을 비교해서 최적의 답변을 종합하고 정리해줘
    의견이 다른 경우는 따로 정리해서 표시해줘

    - GPT 응답 : {gpt}
    - Gemini 응답 : {gemini}

    - 두 모델의 다른 결론 (없다면 미 표시)

    - 최종 결론
    """
)

final_chain = chain | final_prompt | gpt_model2 | StrOutputParser()

answer = final_chain.invoke({"question" : "ASI에 대해 어떻게 생각해 (긍정/부정 선택)"})

display(Markdown(answer))

## 두 AI 응답 비교 및 종합 정리

### 1. 공통점

두 응답 모두 인공지능을 **단순히 긍정적 또는 부정적으로만 판단하기 어렵다**는 입장을 취하고 있습니다.

공통적으로 강조한 내용은 다음과 같습니다.

- 인공지능은 큰 잠재력을 가지고 있음
- 업무 효율성, 기술 혁신, 사회 문제 해결에 기여할 수 있음
- 동시에 윤리적 문제, 오용 가능성, 사회적 부작용이 존재함
- 따라서 균형 잡힌 시각과 신중한 관리가 필요함

즉, 두 응답 모두 **“AI는 가능성과 위험을 동시에 가진 기술이며, 책임 있는 개발과 활용이 중요하다”**는 방향으로 정리할 수 있습니다.

---

### 2. GPT 응답의 특징

GPT 응답은 **일반적인 인공지능 AI 전반**에 대해 설명하고 있습니다.

#### 장점
- 중립적이고 균형 잡힌 관점을 제시함
- 긍정적 측면과 부정적 측면을 간단명료하게 정리함
- 의료, 교통, 환경, 업무 효율성 등 현실적인 활용 분야를 언급함

#### 한계
- 내용이 다소 일반적임
- ASI, 즉 인공 초지능에 특화된 위험성이나 통제 문제는 깊게 다루지 않음
- 명확한 입장 표현보다는 중립적 설명에 가까움

---

### 3. Gemini 응답의 특징

Gemini 응답은 **ASI, 즉 인공 초지능**에 초점을 맞추고 있습니다.

#### 장점
- ASI의 잠재력과 위험성을 구체적으로 설명함
- 기후 변화, 난치병, 빈곤, 에너지 문제 등 거대한 인류 문제 해결 가능성을 언급함
- 정렬 문제, 통제 불능, 실존적 위협 등 ASI 특유의 핵심 위험을 다룸
- “신중한 낙관론”이라는 비교적 명확한 입장을 제시함

#### 한계
- 일반 AI가 아니라 ASI에 초점이 맞춰져 있어, 질문이 일반 AI에 대한 것이라면 범위가 다소 좁거나 과장되어 보일 수 있음
- 실존적 위협 등은 중요한 논점이지만, 현재의 AI 활용 문제와는 구분해서 봐야 함

---

## 두 모델의 다른 결론

### 차이점 1: 다루는 대상의 범위가 다름

- **GPT**는 일반적인 인공지능 전반을 다룸
- **Gemini**는 인공 초지능, 즉 ASI를 중심으로 다룸

따라서 두 응답은 완전히 같은 주제에 대한 답이라기보다는,  
**GPT는 현재와 가까운 AI**, **Gemini는 미래의 고도화된 초지능 AI**에 더 초점을 둔 답변이라고 볼 수 있습니다.

---

### 차이점 2: 입장 표현의 강도

- **GPT**는 “감정이나 주관적 견해가 없다”며 중립적 설명에 머묾
- **Gemini**는 “신중한 낙관론”이라는 입장을 제시함

즉, Gemini가 더 명확한 관점을 제시했고, GPT는 더 조심스럽고 중립적인 태도를 보였습니다.

---

### 차이점 3: 위험성의 수준

- **GPT**는 프라이버시, 실업, 오용 가능성 등 현실적이고 현재적인 문제를 언급함
- **Gemini**는 통제 불능, 정렬 문제, 실존적 위협 등 장기적이고 극단적인 위험까지 다룸

따라서 Gemini 응답이 위험성을 더 깊고 강하게 다룬다고 볼 수 있습니다.

---

## 종합한 최적의 답변

인공지능, 특히 고도화된 인공지능이나 ASI에 대해 단순히 긍정적 또는 부정적이라고 단정하기는 어렵습니다. 인공지능은 인류에게 매우 큰 기회를 제공할 수 있지만, 동시에 심각한 위험과 윤리적 문제도 동반하기 때문입니다.

긍정적인 측면에서 AI는 업무 효율성을 높이고, 과학 기술 발전을 가속화하며, 의료·교통·환경·에너지·교육 등 다양한 분야에서 혁신을 이끌 수 있습니다. 더 발전된 ASI가 등장한다면 기후 변화, 난치병, 빈곤, 에너지 위기처럼 인간이 해결하기 어려웠던 복잡한 문제에 새로운 해결책을 제시할 가능성도 있습니다.

반면 부정적인 측면도 분명합니다. 현재의 AI는 개인정보 침해, 편향된 판단, 일자리 대체, 허위정보 확산, 악용 가능성 등의 문제를 낳을 수 있습니다. 더 나아가 ASI 수준의 기술에서는 인간의 가치와 목표에 맞게 AI를 통제하고 정렬하는 문제가 매우 중요해집니다. 만약 AI가 인간의 의도와 어긋나게 작동하거나 특정 집단에 의해 악용된다면, 사회적 혼란뿐 아니라 인류 전체에 심각한 위협이 될 수도 있습니다.

따라서 가장 적절한 입장은 **신중한 낙관론**이라고 볼 수 있습니다. AI의 발전 가능성은 긍정적으로 바라보되, 그 위험성을 과소평가해서는 안 됩니다. AI 개발과 활용에는 윤리적 기준, 안전 장치, 투명성, 책임성, 법적 규제, 국제적 협력이 함께 따라야 합니다.

---

## 최종 결론

AI는 인류에게 큰 이익을 가져올 수 있는 강력한 기술이지만, 통제와 책임 없이 발전할 경우 큰 위험이 될 수도 있습니다.  
따라서 AI에 대한 최선의 태도는 **무조건적인 낙관도, 막연한 비관도 아닌 “신중한 낙관론”**입니다.

즉, AI의 가능성은 적극적으로 활용하되, 안전성·윤리성·사회적 영향에 대한 관리와 규제를 병행해야 합니다.

# 사용 토큰량 확인

In [87]:
llm1 = init_chat_model(model = "gpt-4o-mini")
llm2 = init_chat_model(model = "gpt-5.4")

prompt = ChatPromptTemplate.from_template(
    "{target}와 갈 수 있는 {region}의 대표적인 {place} 3곳을 추천해줘")

chain1 = prompt | llm1 | StrOutputParser()
chain2 = prompt | llm2 | StrOutputParser()

#토큰 및 요금 분석
with get_openai_callback() as cb1 :
    answer1 = chain1.invoke({"target" : "30살 아내", "region" : "광주", "place" : "음식점"})
with get_openai_callback() as cb2 :
    answer2 = chain2.invoke({"target" : "30살 아내", "region" : "광주", "place" : "음식점"})

    display(Markdown(answer1))
    display(Markdown(answer2))
    print(cb1)
    print(cb2)

광주에는 맛있는 음식점이 많이 있습니다. 30살 아내와 함께 가기 좋은 대표적인 음식점 3곳을 추천해드릴게요.

1. **수정식당**  
   - **추천 메뉴**: 국밥
   - **소개**: 수정식당은 광주에서 유명한 국밥집으로, 진한 국물과 부드러운 고기가 특징입니다. 아늑한 분위기에서 정성껏 만든 한 끼를 즐길 수 있습니다.

2. **청기와 대구탕**  
   - **추천 메뉴**: 대구탕
   - **소개**: 신선한 대구를 사용한 대구탕이 인기 있는 곳으로, 얼큰하면서도 깊은 맛이 일품입니다. 건강한 음식으로 아내와 함께 따뜻한 한 끼를 나누기에 좋습니다.

3. **명동칼국수**  
   - **추천 메뉴**: 칼국수
   - **소개**: 쫄깃한 면발과 시원한 국물의 칼국수로 유명한 곳입니다. 다양한 반찬도 제공되어 여러 가지 맛을 즐길 수 있습니다. 편안한 분위기에서 식사하기에 좋은 자리입니다.

맛있는 음식과 함께 즐거운 시간 보내시길 바랍니다!

광주에서 30살 아내분과 함께 가기 좋은 **대표적인 음식점 3곳**을 분위기와 지역성까지 고려해서 추천드릴게요.  
너무 관광지 느낌만 나는 곳보다, **광주다운 맛 + 데이트하기 괜찮은 곳** 위주로 골랐습니다.

---

## 1. **송정떡갈비**
- **대표 메뉴:** 떡갈비, 한우떡갈비, 육회비빔밥
- **추천 이유:**  
  광주 하면 가장 먼저 떠오르는 음식 중 하나가 **떡갈비**인데, 그 대표격으로 많이 찾는 곳입니다.  
  고기가 부드럽고 달짝지근한 양념이 잘 배어 있어서 **호불호가 적고**, 부부가 편하게 즐기기 좋아요.
- **이런 분께 추천:**  
  광주다운 대표 메뉴를 깔끔하게 먹고 싶을 때
- **분위기:**  
  너무 캐주얼하지도, 너무 무겁지도 않아서 **데이트 식사**로 무난합니다.

---

## 2. **영미오리탕**
- **대표 메뉴:** 오리탕
- **추천 이유:**  
  광주를 대표하는 **지역 음식** 중 하나가 바로 오리탕입니다.  
  들깨와 미나리, 얼큰한 국물 조합이 매력적이고 다른 지역에서는 이 스타일이 흔하지 않아서 **“광주에서만 느낄 수 있는 맛”**으로 추천할 만합니다.
- **이런 분께 추천:**  
  평범한 고깃집보다 좀 더 **광주다운 특별한 음식**을 먹고 싶을 때
- **분위기:**  
  맛집 느낌이 강한 곳이라 세련된 레스토랑 분위기보다는 **현지 인기 맛집**에 가깝습니다.  
  그래도 함께 색다른 음식 경험하기엔 아주 좋아요.

---

## 3. **엄마네돼지찌개**
- **대표 메뉴:** 돼지찌개
- **추천 이유:**  
  광주 현지인들에게 워낙 유명한 곳으로, 칼칼하고 진한 국물 맛이 인상적입니다.  
  밥 한 공기 뚝딱하게 되는 스타일이라, **든든하고 만족감 높은 한 끼**를 원할 때 좋아요.
- **이런 분께 추천:**  
  여행 중 진짜 현지 맛집 느낌을 원할 때
- **분위기:**  
  소박하고 활기찬 편이라, 너무 조용한 데이트보다는 **맛 중심의 데이트**에 잘 맞습니다.

---

# 아내분과 가기 좋게 정리하면
- **광주 대표 음식 처음 먹어본다면:** **송정떡갈비**
- **가장 지역색 강한 메뉴를 원하면:** **영미오리탕**
- **현지인 맛집 감성으로 든든하게 먹고 싶다면:** **엄마네돼지찌개**

---

# 추가 팁
30살 부부 데이트 느낌으로는 보통 이렇게 많이 만족합니다:
- **점심:** 송정떡갈비  
- **저녁:** 영미오리탕 또는 분위기 좋은 카페 코스 추가

원하시면 제가 이어서  
**“광주에서 아내와 가기 좋은 분위기 좋은 식당 3곳”**  
또는  
**“광주 1박 2일 맛집+카페 데이트 코스”**  
형태로 더 추천해드릴게요.

Tokens Used: 313
	Prompt Tokens: 28
		Prompt Tokens Cached: 0
	Completion Tokens: 285
		Reasoning Tokens: 0
Successful Requests: 1
Total Cost (USD): $0.00017519999999999998
Tokens Used: 818
	Prompt Tokens: 27
		Prompt Tokens Cached: 0
	Completion Tokens: 791
		Reasoning Tokens: 0
Successful Requests: 1
Total Cost (USD): $0.0
